# 1049. Last Stone Weight II

## Topic Alignment
- This 0/1 knapsack minimization problem models resource balancing scenarios like load balancing across servers, portfolio optimization minimizing risk variance, or partitioning computational tasks to minimize idle time.

## Metadata 摘要
- Source: https://leetcode.com/problems/last-stone-weight-ii/
- Tags: Dynamic Programming, Array, 0/1 Knapsack
- Difficulty: Medium
- Priority: Medium

## Problem Statement 原题描述
You are given an array of integers `stones` where `stones[i]` is the weight of the `i-th` stone.

We are playing a game with the stones. On each turn, we choose any two stones and smash them together. Suppose the stones have weights `x` and `y` with `x <= y`. The result of this smash is:

- If `x == y`, both stones are destroyed, and
- If `x != y`, the stone of weight `x` is destroyed, and the stone of weight `y` has new weight `y - x`.

At the end of the game, there is **at most one** stone left.

Return the **smallest possible weight** of the left stone. If there are no stones left, return `0`.

**Constraints**:
- 1 <= stones.length <= 30
- 1 <= stones[i] <= 100

## Progressive Hints
- Hint 1: Assign each stone a +/- sign. The problem becomes: minimize |sum of positive - sum of negative|.
- Hint 2: This is equivalent to partitioning stones into two groups with minimal difference.
- Hint 3: If we find the subset closest to total_sum/2, the difference is minimized.
- Hint 4: Use 0/1 knapsack to find the maximum sum ≤ total_sum/2.
- Hint 5: Answer is total_sum - 2 * (maximum achievable sum ≤ total_sum/2).

## Solution Overview
**Key Insight**: This is **partition with minimum difference**, which reduces to **0/1 knapsack**.

**Transformation**:
- Assign each stone to group A (+) or group B (-)
- Final weight = |sum(A) - sum(B)|
- Since sum(A) + sum(B) = total_sum, we have:
  - sum(A) = S_A
  - sum(B) = total_sum - S_A
  - Difference = |S_A - (total_sum - S_A)| = |2×S_A - total_sum|
- To minimize this, we want S_A as close to total_sum/2 as possible

**Problem becomes**: Find the largest sum ≤ total_sum/2 using subset of stones.

**Approach**: 0/1 knapsack with target = total_sum/2, maximize achievable sum.

## Detailed Explanation

### From Stone Smashing to Subset Partition

**Observation**: The smashing operation is equivalent to:
- Assign + to one group of stones
- Assign - to another group
- Final result = |sum of + group - sum of - group|

**Example**: stones = [2, 7, 4, 1, 8, 1]
- One way: (+2 +7 +4) vs (-1 -8 -1) = 13 - 10 = 3
- Another: (+8 +1 +1) vs (-2 -7 -4) = 10 - 13 = |-3| = 3
- Optimal: (+7 +4 +1) vs (-2 -8 -1) = 12 - 11 = 1

---

### Mathematical Transformation

Let:
- S = total_sum of all stones
- A = sum of positive group
- B = sum of negative group
- A + B = S

**Goal**: Minimize |A - B|

**Substitute**:
- B = S - A
- |A - B| = |A - (S - A)| = |2A - S|

**To minimize |2A - S|**:
- We want A as close to S/2 as possible
- Best A is the largest value ≤ S/2

**Answer**: 
```
min_difference = S - 2 × A_max
```
where A_max = maximum achievable sum ≤ S/2

---

### 0/1 Knapsack to Find Maximum Sum ≤ Target

**Standard 0/1 knapsack**:
```python
dp[capacity] = max value achievable with capacity limit
```

**Our variant**:
```python
dp[sum] = can we achieve this sum?
```
or
```python
dp[sum] = True if achievable, False otherwise
```

After DP, find the largest sum ≤ S/2 where dp[sum] = True.

---

### Algorithm Steps

**Step 1**: Calculate total sum S and target = S // 2

**Step 2**: Initialize DP
- `dp[0] = True` (can achieve sum 0)
- `dp[i] = False` for i > 0

**Step 3**: For each stone:
- Traverse from target down to stone (right to left, 0/1 knapsack)
- If `dp[i - stone]` is True, set `dp[i] = True`

**Step 4**: Find maximum sum ≤ target where dp[sum] = True

**Step 5**: Return S - 2 × max_sum

---

### Example Walkthrough

**Input**: stones = [2, 7, 4, 1, 8, 1]

**Step 1**: S = 23, target = 11

**Step 2**: `dp = [True, False, False, ..., False]` (size 12)

**Step 3**: Process each stone
- After stone=2: dp[2] = True
- After stone=7: dp[2], dp[7], dp[9] = True
- After stone=4: dp[2], dp[4], dp[6], dp[7], dp[9], dp[11] = True
- Continue...

**Step 4**: Maximum sum ≤ 11 where dp[sum] = True is 11

**Step 5**: Answer = 23 - 2×11 = 1

**Verification**: Partition [7,4,1] (sum=12) vs [2,8,1] (sum=11), difference = 1 ✓

---

### Why This is 0/1 Knapsack

- **Items**: stones
- **Capacity**: total_sum / 2
- **Item weight**: stone value
- **Item value**: stone value (same as weight)
- **Goal**: Maximize total value ≤ capacity
- **Constraint**: Each stone used at most once (0/1 property)

**Template**:
```python
for stone in stones:                     # Outer: items
    for capacity in range(target, stone-1, -1):  # Inner: capacity (RIGHT TO LEFT)
        dp[capacity] = dp[capacity] or dp[capacity - stone]
```

## Complexity Trade-off Table
| Approach | Time | Space | Notes |
| --- | --- | --- | --- |
| Backtracking | O(2^n) | O(n) | Too slow |
| 0/1 Knapsack DP | O(n × sum) | O(sum) | Most efficient |
| Meet-in-middle | O(2^(n/2) × log(2^(n/2))) | O(2^(n/2)) | Good for larger n |

In [ ]:
class Solution:
    def lastStoneWeightII(self, stones: list[int]) -> int:
        """
        0/1 Knapsack solution - find partition with minimum difference.
        
        Time: O(n × sum)
        Space: O(sum)
        """
        total_sum = sum(stones)
        target = total_sum // 2
        
        # dp[i] = can we achieve sum i?
        dp = [False] * (target + 1)
        dp[0] = True  # Can always achieve sum 0
        
        # 0/1 knapsack: right to left traversal
        for stone in stones:
            for i in range(target, stone - 1, -1):
                # If we can make (i - stone), we can make i
                dp[i] = dp[i] or dp[i - stone]
        
        # Find maximum achievable sum <= target
        max_sum = 0
        for i in range(target, -1, -1):
            if dp[i]:
                max_sum = i
                break
        
        # Minimum difference = total_sum - 2 * max_sum
        return total_sum - 2 * max_sum

In [ ]:
# Test cases
tests = [
    ([2, 7, 4, 1, 8, 1], 1),    # Partition: [7,4,1] vs [2,8,1]
    ([31, 26, 33, 21, 40], 5),  # Check partition
    ([1, 2], 1),                # Simple case
    ([1], 1),                   # Single stone
    ([1, 1], 0),                # Two equal stones
    ([2, 2, 2, 2], 0),          # All equal
    ([100], 100),               # Single large stone
]

solver = Solution()
for stones, expected in tests:
    result = solver.lastStoneWeightII(stones)
    assert result == expected, f"Failed for stones={stones}: got {result}, expected {expected}"
print('All tests passed.')

## Complexity Analysis
- **Time**: O(n × sum) where n = len(stones), sum = sum(stones)
  - For each stone, iterate through up to sum/2 values
  - With constraints: n ≤ 30, sum ≤ 3000, so O(30 × 1500) = O(45,000)
- **Space**: O(sum)
  - DP array of size (sum/2 + 1)
  - At most O(1500) for this problem

## Edge Cases & Pitfalls
- **Single stone**: Return the stone weight (no partition possible)
- **Two equal stones**: Return 0 (they cancel out)
- **All equal stones**: If even count, return 0; if odd, return stone_value
- **Odd total sum**: Can never partition exactly equally, but DP handles this
- **Finding max_sum**: Must search from target down to 0 to find largest achievable sum
- **Traversal direction**: MUST traverse right to left for 0/1 knapsack
- **Formula**: Answer is total_sum - 2×max_sum, not max_sum - (total_sum - max_sum)

## Follow-up Variants
- **Exact partition**: Return whether can partition into two exactly equal subsets (LC 416)
- **K-way partition**: Partition into k groups with minimum max difference
- **Weighted stones**: Stones have both weight and value, different optimization criteria
- **With operations**: Can split stones or combine them differently
- **Online version**: Stones arrive in stream, maintain optimal partition
- **Print partition**: Return the actual partition, not just minimum difference

## Takeaways
- **Problem Transformation**: Stone smashing ⇔ Partition with minimum difference ⇔ 0/1 knapsack
- **Mathematical Insight**: minimize |A - B| = minimize |2A - S| where A ≤ S/2
- **Formula**: Answer = total_sum - 2 × (max achievable sum ≤ total_sum/2)
- **0/1 Knapsack Pattern**: Right-to-left traversal ensures each stone used at most once
- **Finding Maximum**: After DP, search from target down to find largest achievable sum
- **Relation to LC 416**: This generalizes LC 416 from "can partition equally?" to "minimum partition difference"
- **Optimization Goal**: Unlike LC 322 (minimize count), here we maximize sum within capacity

## Similar Problems
| Problem ID | Problem Title | Technique |
| --- | --- | --- |
| LC 416 | Partition Equal Subset Sum | 0/1 knapsack boolean |
| LC 494 | Target Sum | 0/1 knapsack counting |
| LC 698 | Partition to K Equal Sum Subsets | Backtracking + DP |
| LC 805 | Split Array With Same Average | Complex partition problem |
| LC 1046 | Last Stone Weight | Heap-based simulation |